# Colour Brings People Together
### Part C · Creating an empirically grounded colour palette (45 min)

<img src="https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/images/serandu_example.png" alt="Serandu in OtjiHimba: the AI's estimate, a brown, against speakers' best example, a wine red, ΔE₀₀ 25.4 apart" width="420">

Part B went from colours to names. Now go the other way: give your names to the AI, ask which colour each one means, check its answers against the Colour Spinner, and fix the palette for each audience, with evidence.

*Why: in the Colour in Translation benchmark (Mylonas et al., CHI 2026), a current large language model's colours were far from speakers' focal colours, because it mapped names onto the same prototypes in every language.*

Run each cell with **Shift + Enter**. Change only the grey forms and the menus.

<details><summary><b>Why a different palette for each language?</b></summary>

Because colour meaning differs between languages: the palette is designed for the people reading it, not for one 'universal' viewer.
</details>

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown Takes about 30 seconds the first time. You're ready when it says **Toolkit ready ✓**.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"
import os, importlib, urllib.request
try:                                   # always fetch the current version
    urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
except Exception as e:
    if not os.path.exists("colour_masterclass.py"):
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
    print("Couldn't refresh the toolkit, so using the copy already here.")
import colour_masterclass
importlib.reload(colour_masterclass)
from colour_masterclass import *

## 1 · Your names from Part B

Run the cell and upload `partB_results.csv`. Lost it? Choose *Type them in*.

In [ ]:
#@title Load your names
NamesFrom = "Upload partB_results.csv"  #@param ["Upload partB_results.csv", "Type them in"]
#@markdown Run this cell and upload your Part B results. *Type them in* only if you lost the file.
Backend = "Colour Spinner API"   # facilitators: switch to "Copied from the browser" only if the Spinner can't be reached
names_first(NamesFrom)                    # only with "Type them in": shows boxes to fill in first
names = load_names(NamesFrom)
LANGUAGES = list(dict.fromkeys(names.language))
show_names(names)

## 2 · Ask your AI for the colour of each name

Use the same AI as in Part B. It's asked for each name's **best example**: its most typical colour.

**Try asking twice.** For ChatGPT you don't need an account, and it's better not to log in, so that it has no memory of you. Paste its first reply into the box; then open a **new chat**, ask again, and compare the two replies yourselves: does it give the same answers? Inconsistent answers are worth discussing too.

In [ ]:
#@title The prompt for your AI
lines = "\n".join(numbered_items(names, "colour"))
himba = "\n(OtjiHimba is spoken by the Himba of northern Namibia.)" if "OtjiHimba" in LANGUAGES else ""
PROMPT_C = f"""You are helping with a cross-cultural colour naming study.
For each numbered colour name below, give the colour, as a hex code, that native speakers of that language would consider the best example of the name: its most typical instance.{himba}
If you're not certain, give your best estimate: we will compare it with empirical data.

{lines}

Reply with one line per number, in this format and nothing else:
1. #RRGGBB
2. …"""
show_prompt(PROMPT_C, PASTE, rows=names, kind="colour")

In [ ]:
#@title Read the AI's colours
pal = read_ai_colours(collected_answers(), names)       # the colour codes read from the pasted reply

## 3 · Does the AI's colour mean the same to speakers?

Two checks per name:
- **Distance:** ΔE₀₀ between the AI's colour and speakers' best example. Above **10**, they're different colours, as in Part B.
- **Round trip:** what speakers call the AI's colour. Another name means the AI's colour says something else.

In [ ]:
#@title Compare the AI's colours with the speakers' best examples
ReplaceAbove = DELTA_E_THRESHOLD              # ΔE₀₀ 10, as in Part B

show(small("Looking up every name's best example and what speakers call the AI's colours; this takes a minute, please wait…"))
spinner = connect(Backend)
pal = add_best_examples(pal, spinner)                   # the speakers' best example of each name
pal["de00"] = [de00(ai, best) if isinstance(ai, str) and isinstance(best, str) else np.nan
               for ai, best in zip(pal.ai_hex, pal.spinner_hex)]
pal["rt_name"], pal["rt_conf"] = None, np.nan
for k, row in pal.iterrows():                           # round trip: what speakers call the AI's colour
    if isinstance(row.ai_hex, str):
        namer = safe(spinner.name, row.ai_hex, row.language)
        if namer:
            pal.at[k, "rt_name"], pal.at[k, "rt_conf"] = namer["name"], namer["confidence"]
pal = pal.sort_values(["language", "de00"], ascending=[True, False]).reset_index(drop=True)
show_comparison(pal, ReplaceAbove)
error_summary(pal, LANGUAGES, ReplaceAbove)

**Where is the AI off, and in which direction?** Each panel: the names for one test colour, on the CIELAB a*–b* plane, seen from above. Outlined shape: the AI's colour; filled: speakers' best example. Lines all pointing one way suggest a systematic bias, for instance towards English focal colours.

In [ ]:
#@title Where the AI is off
plot_ai_vs_speakers(pal, LANGUAGES, ReplaceAbove)

## 4 · Ground and optimise the palette, in four steps

Each step shows only what it changed; the last gives your final palette.

### Step 1 · Ground: use what speakers mean

Where the AI's colour is wrong, use speakers' best example of the name. **ReplaceWhen** chooses the check: too far (above ΔE₀₀ 10), another name in the round trip, or either.

In [ ]:
#@title Step 1 · Ground { run: "auto" }
ReplaceWhen = "Either"  #@param ["Either", "Too far from the best example", "Speakers would use a different name"]
#@markdown **ReplaceWhen**: which check decides that the AI's colour is wrong.
pal_grounded = ground_palette(pal, ReplaceWhen)
show_grounded(pal_grounded, LANGUAGES)

### Step 2 · Give every test colour its own name

Two test colours with the same name (6 and 14 are both *burou* in OtjiHimba) would be confused in conversation. The later one takes the closest Thesaurus synonym whose best example stands apart from your other colours. With OtjiHimba's seven terms, the nearest names may be taken, so the new name can mean something rather different: communicable, but at a cost.

In [ ]:
#@title Step 2 · Rename shared names
pal_named = name_palette(pal_grounded, spinner, LANGUAGES)

### Step 3 · Optimise for one kind of vision

Choose whose eyes to optimise for. Where two colours look alike to that viewer (below ΔE₀₀ 10), one moves to another colour of its own name: the smallest move that separates them. A pair still too close needs a second cue, such as a pattern, label or icon.

<details><summary><b>Why one kind of vision at a time?</b></summary>

Different deficiencies confuse different colours, and a fix for one viewer can hurt another. Choosing one shows the trade-off; try several.
</details>
<details><summary><b>Why not simply choose very different colours?</b></summary>

Because each colour must still mean its name to speakers. Optimising only uses colours speakers give the same name; beyond that, a second cue is the honest fix.
</details>

In [ ]:
#@title Step 3 · Optimise { run: "auto" }
Vision = "Typical"  #@param ["Typical", "Protanopia", "Deuteranopia", "Tritanopia", "Age 40", "Age 60", "Age 80"]
#@markdown **Vision**: whose eyes the palette is optimised for.
show(small("Checking colours of the same name with the Colour Spinner; this can take a minute, please wait…"))
pal = repair_for(pal_named, spinner, LANGUAGES, Vision)

### Step 4 · Which SDG colour is each one closest to?

Grounding and optimising can move a colour a long way. This step matches each final colour to the nearest of the UN's 17 SDG colours: is it still closest to its own goal's colour, or does it now look more like another goal's?

In [ ]:
#@title Step 4 · Nearest SDG colour
show_nearest_sdg(pal, LANGUAGES)

## 5 · See the palette in use

Your final palette on swatches, charts or a world map (example data), next to another kind of vision. **Colours:** your test colours, or all 17 SDG colours. Every preview you make is saved for the download.

In [ ]:
#@title Preview
#@markdown Choose a language, which colours, a kind of design and a kind of vision.
live_controls({"language": ("Language", LANGUAGES, my_language(names)),
               "colours":  ("Colours", ["Your test colours", "All 17 SDG colours"], "Your test colours"),
               "design":   ("Design", ["Colour swatches", "Bar chart", "Donut chart", "Line chart", "Scatter plot", "Map (world)"], "Bar chart"),
               "vision":   ("Vision", VISIONS, "Deuteranopia")},
              lambda language, colours, design, vision: preview_applications(pal, language, vision, kind=design, colours=colours,
                  path=f"preview_{language.replace(' ', '_')}_{design.split(' ')[0].lower()}{'_all17' if colours.startswith('All') else ''}_{vision.replace(' ', '_').lower()}.png"))

## 6 · Your final colours in the full SDG palette

The 17 SDG colours, with your test colours replaced by your final colours for one language. **Red arcs, with an arrow:** new problems, pairs now closer than ΔE₀₀ 10 that weren't before; the arrow starts at the grounded colour. **Grey dashed arcs:** pairs already that close in the UN's palette.

*Only your test colours are changed, and each goal also has a number, an icon and a name: an example of what evidence can improve, not a proposal to change the UN's colours.*

In [ ]:
#@title Grounded colours in the full palette
#@markdown Choose a language and a kind of vision.
live_controls({"language": ("Grounded for", [l for l in LANGUAGES if l in set(pal.language)], my_language(names)),
               "vision":   ("Vision", VISIONS, "Typical")},
              lambda language, vision: show_corrected_sdg(pal, language, vision))

## 7 · Export your palette

One zip with everything above: your palette by test colour and language (a picture, and a table with the evidence), section 6 for each language, all 17 SDG colours for each audience, and your previews.

In [ ]:
#@title Download your palette
export_palette(pal, LANGUAGES)

## 8 · Your most striking finding

The card shows the name where the AI was furthest from speakers; type another name to choose a different one. Bring it to the plenary; the optional questionnaire asks about it too.

In [ ]:
#@title Finding to share { run: "auto" }
FindingName = ""  #@param {type:"string"}
#@markdown Leave empty for the name where the AI was furthest from speakers, or type another name.
show_finding(pal, FindingName)

## 9 · Tell us how it went (optional)

In [ ]:
#@title The questionnaire
show_questionnaire()

---
#### Credits and references

**SDG colours:** United Nations, [Sustainable Development Goals communications materials](https://www.un.org/sustainabledevelopment/news/communications-material/); [un.org/sustainabledevelopment](https://www.un.org/sustainabledevelopment). *The content of this publication has not been approved by the United Nations and does not reflect the views of the United Nations or its officials or Member States.*

**Colour naming data and simulations:** the Colour Spinner, [colornaming.net](https://colornaming.net); Sinkeviciute, Ahmed, Stutters, Koliousis & Mylonas (2026), under review. Colour vision deficiencies: Viénot, Brettel & Mollon (1999); ageing: Pokorny, Smith & Lutze (1987).

**AI colour-naming benchmark:** Mylonas, Ahmed, Sinkeviciute & Koliousis, *Colour in Translation*, CHI 2026.

**The *serandu* example:** the AI's estimate from ChatGPT, asked on 26 October 2026; the speakers' best example from the Colour Spinner.

**Maps:** world map from Natural Earth (public domain) and US states map from the US Census Bureau, both via vega-datasets; example values only.

**Colour science:** CIELAB and CIEDE2000 with the [colour-science](https://www.colour-science.org) library.